# PTCG ABC — Colab 実行環境

コードセルには `# [C1]` のような番号を振ってある。実行するセルはこちらから番号で指定する。

**毎回の起動時に必要なのは C1〜C4。** ランタイムが切れると `/content` は消えるので、
再開時も C2〜C4 を流し直すこと。C5 以降は目的に応じて選ぶ。

前提は `docs/design.md` の 0 節と 7 節。

- エンジンは `libcg.so` (Linux) を同梱しているので Colab で動く。Kaggle 本番も Linux
- 乱数シードは指定できない。再現性は対戦ログで担保する
- GPU / TPU を割り当てても速くならない。CUDA を使う処理がない。関心は vCPU 数だけ
- 資源は保証されないので、小分けに回して 1 戦ごとに Drive へ追記する

In [ ]:
# [C1] 環境の確認
import os, platform, subprocess
print(platform.platform(), 'python', platform.python_version())
try:
    cpu = len(os.sched_getaffinity(0))
except AttributeError:
    cpu = os.cpu_count()
print('使える論理 CPU:', cpu)
print(subprocess.run(['free', '-h'], capture_output=True, text=True).stdout)

In [ ]:
# [C2] インストール。Mac 側と同じ版に固定する (版が違うとエンジンの挙動が変わる)
!pip -q install 'kaggle-environments==1.32.2'

In [ ]:
# [C3] リポジトリの取得。Colab のシークレットに GITHUB_TOKEN を登録しておくこと。
#      既にクローン済みなら git pull で更新する。
import os, subprocess, glob
REPO, BRANCH, WORK = 'Tomomon2525/mutsugi_team', 'tkawamura', '/content/ptcg-abc'

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception as e:
    print('シークレット未設定:', e)

if os.path.isdir(os.path.join(WORK, '.git')):
    r = subprocess.run(['git', '-C', WORK, 'pull', '--ff-only'], capture_output=True, text=True)
    print(r.stdout or r.stderr)
elif token:
    url = f'https://{token}@github.com/{REPO}.git'
    r = subprocess.run(['git', 'clone', '--depth', '50', '-b', BRANCH, url, WORK],
                       capture_output=True, text=True)
    print(r.stdout or r.stderr)
else:
    from google.colab import drive
    drive.mount('/content/drive')
    zips = sorted(glob.glob('/content/drive/MyDrive/**/ptcg-abc*.zip', recursive=True))
    assert zips, 'GITHUB_TOKEN も Drive 上の zip も見つからない'
    !unzip -q -o "{zips[-1]}" -d /content/

os.chdir(WORK)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

In [ ]:
# [C4] エンジンの動作確認と、時間予算の設定。C5 以降はこの設定を引き継ぐ。
import sys, os, tarfile, glob
sys.path.insert(0, '/content/ptcg-abc/shared')
import ptcg
from kaggle_environments.envs.cabt.cg import sim
print('共有ライブラリ:', sim.lib._name, '/ カード種数:', len(ptcg.cards()), '/ id=648 ->', ptcg.name(648))

# Kaggle 相当の時間予算 (Mac 基準で較正した値)
os.environ.update(PTCG_TIME_POOL='70', PTCG_MAX_SLICE='0.58',
                  PTCG_RESERVE='5.2', PTCG_MIN_SLICE='0.03')

# 過去バージョンを展開する。submission.tar.gz は shared/*.py を同梱した自己完結物
for tgz in sorted(glob.glob('champions/*/submission.tar.gz')):
    d = os.path.join(os.path.dirname(tgz), 'agent')
    os.makedirs(d, exist_ok=True)
    with tarfile.open(tgz) as t:
        t.extractall(d)
    print('champion 展開:', d)

from google.colab import drive
drive.mount('/content/drive', force_remount=False)
OUT = '/content/drive/MyDrive/ptcg-abc/league'
os.makedirs(OUT, exist_ok=True)
print('結果の保存先:', OUT)

## スループットの較正 (C5)

**戦/時が最大の並列数を選んではいけない。** 思考時間は壁時計で切っているので、
コア数を超えてプロセスを詰め込んでも 1 戦の時間はあまり延びず、代わりに 1 手あたりの
ロールアウト数が減る。数はこなせるがエージェントは弱くなる。

採るのは「**思考量を保てる範囲での最良**」の行である。

In [ ]:
# [C5] 並列数ごとの速度と思考量
!python tools/bench.py -j 1,2,4 -n 3

## 対戦 (C6 以降)

`JOBS` は C5 の「思考量を保てる範囲での最良」に合わせる。
途中で切れても、同じセルをもう一度実行すれば未完了分から再開する。

In [ ]:
# [C6] 並列数の設定。C5 の結果に合わせて書き換える
JOBS = 2
print('JOBS =', JOBS)

In [ ]:
# [C7] 相手のカードを指す専用ルールの切り分け
#      Mac 実測: 現行版は champion に 44.5% (200戦, z=-1.56)、context_signs 単体は
#      中立 (51.0%, 100戦)。残る容疑がこのルール。
!python tools/league.py agents/d0_grimmsnarl agents/d0_nofoe \
    -n 200 -j {JOBS} -b 25 -o {OUT}/foe_ab.jsonl

In [ ]:
# [C8] 現行版と champion。Mac の 44.5% が Colab でも再現するかを見る
!python tools/league.py agents/d0_grimmsnarl champions/572_6/agent \
    -n 200 -j {JOBS} -b 25 -o {OUT}/ctx_vs_champ.jsonl

In [ ]:
# [C9] 対面別。環境で実際に当たる 3 デッキに対する現状把握
for foe in ['e_lucario', 'e_alakazam', 'e_dragapult']:
    print('=' * 60, foe)
    !python tools/league.py agents/d0_grimmsnarl agents/{foe} \
        -n 100 -j {JOBS} -b 25 -o {OUT}/d0_vs_{foe}.jsonl

In [ ]:
# [C10] 集計だけ見る (対戦は回さない)
import glob, json, os
for path in sorted(glob.glob(f'{OUT}/*.jsonl')):
    w = l = d = 0
    for line in open(path):
        try:
            r = json.loads(line)
        except Exception:
            continue
        if 'result' not in r:
            continue
        w += r['result'] > 0
        l += r['result'] < 0
        d += r['result'] == 0
    n = w + l + d
    if not n:
        continue
    z = (w / n - 0.5) / (0.25 / n) ** 0.5
    print(f"{os.path.basename(path):<28} {n:>4}戦 {w:>3}勝{l:>3}敗{d:>3}分  {w/n:>6.1%}  z={z:+.2f}")